In [15]:
import torch
import torch.nn as nn

from src.quadratic_annealing_optimizer import QuadraticAnnealingOptimizer
from src.models import QuadraticMLP
from src.utils import build_sampler, data_load_and_prep
from src.training import train

## Idea

For the current parameter vector $w$, the batch loss is approximated locally by a second-order Taylor expansion:

$$
\mathcal{L}(w + \Delta) \approx \mathcal{L}(w) + g^\top \Delta + \frac{1}{2}\Delta^\top H\Delta,
$$

where $g = \nabla \mathcal{L}(w)$ and $H = \nabla^2 \mathcal{L}(w)$.

The annealer does not optimize the full continuous update directly. Instead, for a selected subset of parameters, it chooses binary variables $z_i \in \{0,1\}$ that encode the sign of a fixed-size step:

$$
\Delta_i = \eta(2z_i - 1),
$$

so $z_i = 1$ means a $+\eta$ step and $z_i = 0$ means a $-\eta$ step. Substituting this into the quadratic model turns the local optimization problem into a QUBO/BQM:

$$
E(z) = \sum_i a_i z_i + \sum_{i<j} b_{ij} z_i z_j + c.
$$

The annealer minimizes $E(z)$, then the proposed update is applied to the network parameters and accepted only if the true loss decreases.

## Loading sample Iris data for training

In [16]:
train_loader, test_loader = data_load_and_prep("iris", test_size=0.3, random_state=42, batch_size='full')

## Training

In [ ]:
loss_fn = nn.CrossEntropyLoss()
model = QuadraticMLP(4, [32, 16], 3)
classical_device = "cuda" 

sampler = build_sampler(mode="simulated")

optimizer = QuadraticAnnealingOptimizer(
    sampler=sampler,
    model=model,
    device="cpu",
    subset_size=12,
    step_size=0.05,
    num_reads=100,
)

Using sampler: simulated


In [18]:
train(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=optimizer,
    c_device=classical_device,
    epochs=30,
    experiment_name = "quadratic-annealer-iris",
)

Epoch 000 | train_loss=1.1328 | train_acc=0.171 | test_loss=1.1133 | test_acc=0.200 | 
Epoch 005 | train_loss=0.8208 | train_acc=0.819 | test_loss=0.8346 | test_acc=0.733 | 
Epoch 010 | train_loss=0.5922 | train_acc=0.848 | test_loss=0.6213 | test_acc=0.778 | 
Epoch 015 | train_loss=0.4320 | train_acc=0.876 | test_loss=0.4874 | test_acc=0.822 | 
Epoch 020 | train_loss=0.3300 | train_acc=0.886 | test_loss=0.4172 | test_acc=0.822 | 
Epoch 025 | train_loss=0.2423 | train_acc=0.933 | test_loss=0.3488 | test_acc=0.844 | 


2026/03/12 18:17:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/12 18:17:05 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.
2026/03/12 18:17:05 WARNING mlflow.utils.requirements_utils: Found torch version (2.10.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torch==2.10.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


Epoch 029 | train_loss=0.1640 | train_acc=0.981 | test_loss=0.2514 | test_acc=0.889 | 


# Classical optimization for benchmarking

In [9]:
loss_fn = nn.CrossEntropyLoss()
adam_model = QuadraticMLP(4, [16, 8], 3)
classical_device = "cuda" 
adam_optimizer = torch.optim.Adam(adam_model.parameters(), 
                             lr=0.01,
                             betas=[0.9, 0.999],
                             )

## Optimization using Adam optimizaer

In [10]:
train(
    model=adam_model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=adam_optimizer,
    c_device=classical_device,
    epochs=30, 
    experiment_name = "adam-optimizer-iris",
)

2026/03/12 17:54:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/12 17:54:22 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.
2026/03/12 17:54:22 WARNING mlflow.utils.requirements_utils: Found torch version (2.10.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torch==2.10.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


Epoch 000 | train_loss=1.0773 | train_acc=0.352 | test_loss=1.0901 | test_acc=0.356 | 
Epoch 005 | train_loss=0.8109 | train_acc=0.857 | test_loss=0.8460 | test_acc=0.778 | 
Epoch 010 | train_loss=0.6139 | train_acc=0.838 | test_loss=0.6688 | test_acc=0.756 | 
Epoch 015 | train_loss=0.4842 | train_acc=0.848 | test_loss=0.5548 | test_acc=0.756 | 
Epoch 020 | train_loss=0.3982 | train_acc=0.876 | test_loss=0.4819 | test_acc=0.800 | 
Epoch 025 | train_loss=0.3369 | train_acc=0.895 | test_loss=0.4335 | test_acc=0.822 | 
Epoch 029 | train_loss=0.2999 | train_acc=0.895 | test_loss=0.4054 | test_acc=0.822 | 


## Optimization using LBFGS-style second-order optimizer

In [11]:
loss_fn = nn.CrossEntropyLoss()
lbfgs_model = QuadraticMLP(4, [16, 8], 3)
classical_device = "cuda" 
lbfgs_optimizer = torch.optim.LBFGS(lbfgs_model.parameters(), 
                              lr=0.01,
                              max_iter=1,
                              history_size=100,
                              )

In [12]:
train(
    model=lbfgs_model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=lbfgs_optimizer,
    c_device=classical_device,
    epochs=30,
    experiment_name = "lbfgs-optimizer-iris",
)

2026/03/12 17:54:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/12 17:54:25 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.
2026/03/12 17:54:25 WARNING mlflow.utils.requirements_utils: Found torch version (2.10.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torch==2.10.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


Epoch 000 | train_loss=1.0704 | train_acc=0.571 | test_loss=1.0866 | test_acc=0.556 | 
Epoch 005 | train_loss=0.6801 | train_acc=0.857 | test_loss=0.7240 | test_acc=0.733 | 
Epoch 010 | train_loss=0.5278 | train_acc=0.905 | test_loss=0.5879 | test_acc=0.822 | 
Epoch 015 | train_loss=0.4551 | train_acc=0.933 | test_loss=0.5250 | test_acc=0.800 | 
Epoch 020 | train_loss=0.3884 | train_acc=0.962 | test_loss=0.4678 | test_acc=0.800 | 
Epoch 025 | train_loss=0.3299 | train_acc=0.981 | test_loss=0.4154 | test_acc=0.822 | 
Epoch 029 | train_loss=0.2918 | train_acc=0.981 | test_loss=0.3786 | test_acc=0.867 | 
